# 01 Dataset QC and EDA

Dataset: TDC Caco2_Wang public benchmark. Endpoint: experimentally measured Caco-2 log(Papp), an in vitro permeability proxy. This notebook is for early discovery QSAR screening and education only; it does not support clinical claims.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import matplotlib.pyplot as plt
import pandas as pd

from adme_predictor.data import load_caco2_wang_processed, get_caco2_wang_metadata
from adme_predictor.features import DESCRIPTOR_KEYS, calculate_descriptors

FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Load and Document Dataset

Rows are validated by RDKit, canonicalized, deduplicated by canonical SMILES, and labeled using the median log(Papp). The median class is a modeling convenience, not a biological or clinical cutoff.

In [ ]:
df = load_caco2_wang_processed()
metadata = get_caco2_wang_metadata(sample_size=len(df))
display(pd.DataFrame([metadata.__dict__]).T.rename(columns={0: 'value'}))
display(df.head())
print(f'Processed sample size: {len(df)}')

## Missingness and Target Distribution

Missing endpoint values would weaken supervised learning because the model needs paired SMILES and experimental measurements. Outliers should be inspected because permeability assays can vary across protocols and compounds.

In [ ]:
display(df.isna().sum().to_frame('missing_count'))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(df['caco2_log_papp'], bins=30, edgecolor='black')
axes[0].set_title('Caco-2 log(Papp) Distribution')
axes[0].set_xlabel('log(Papp)')
axes[0].set_ylabel('Count')
axes[1].boxplot(df['caco2_log_papp'], vert=True)
axes[1].set_title('Caco-2 log(Papp) Boxplot')
axes[1].set_ylabel('log(Papp)')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_target_distribution.png', dpi=200)
plt.show()

In [ ]:
class_counts = df['permeability_class'].value_counts().sort_index()
ax = class_counts.plot(kind='bar', figsize=(5, 4), title='Median-Threshold Class Balance')
ax.set_xlabel('0 = lower permeability, 1 = higher permeability')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_class_balance.png', dpi=200)
plt.show()
display(class_counts.to_frame('count'))

## Descriptor Generation

Descriptors such as TPSA, HBD/HBA, logP, molecular weight, and rotatable bonds are chemically relevant to passive permeability because they summarize polarity, hydrogen bonding, lipophilicity, size, and flexibility.

In [ ]:
descriptor_df = pd.DataFrame([calculate_descriptors(s) for s in df['canonical_smiles']])
eda_df = pd.concat([df.reset_index(drop=True), descriptor_df], axis=1)
display(eda_df[list(DESCRIPTOR_KEYS) + ['caco2_log_papp', 'permeability_class']].describe().T)

descriptor_df[list(DESCRIPTOR_KEYS)].hist(figsize=(14, 12), bins=30, edgecolor='black')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_descriptor_histograms.png', dpi=200)
plt.show()

In [ ]:
corr = eda_df[list(DESCRIPTOR_KEYS) + ['caco2_log_papp']].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr.index)), corr.index)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Descriptor Correlation Heatmap')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_correlation_heatmap.png', dpi=200)
plt.show()
display(corr['caco2_log_papp'].sort_values(ascending=False).to_frame('correlation_with_log_papp'))

## Scientific Interpretation Checklist

- Class balance should be checked before trusting accuracy; balanced accuracy and AUROC are more informative when classes are uneven.
- High TPSA and many HBD/HBA often indicate polarity and hydrogen bonding that may reduce passive permeability.
- Higher logP can support membrane partitioning, but excessive lipophilicity can introduce solubility and assay artifacts.
- Molecular weight and rotatable bonds capture size and flexibility; larger, flexible molecules may cross membranes less readily.
- Correlations are descriptive, not causal, and this random-split baseline should later be compared against scaffold splits.